## Descarga de datos de la API de Google Maps

In [3]:
import googlemaps
import os
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv

# 1. Configuración Inicial
load_dotenv()
api_key = os.getenv('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=api_key)

In [14]:
import googlemaps
import os
import time
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv

# 1. Configuración Inicial
load_dotenv()
api_key = os.getenv('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=api_key)

# --- CONFIGURACIÓN DE PUNTOS CLAVE ---
origen_query = "Terminal de Ómnibus Mariano Moreno, Rosario"
# Radio central para la búsqueda (Echesortu)
radio_query = "MO Pet Shopping - Sucursal Echesortu"

# Geocodificamos el origen para obtener su Place ID y Coordenadas
res_origen_geo = gmaps.geocode(origen_query)
origen_id = res_origen_geo[0]['place_id']
origen_coords = res_origen_geo[0]['geometry']['location']

# Geocodificamos el centro del radio de búsqueda
res_radio = gmaps.geocode(radio_query)
location_busqueda = res_radio[0]['geometry']['location']

# 2. Definición de Categorías
categorias_busqueda = [
    {'label': 'Restaurante', 'type': 'restaurant', 'keyword': 'restaurant'}, 
    {'label': 'Museo', 'type': 'museum', 'keyword': 'museo'},
    {'label': 'Heladería', 'type': None, 'keyword': 'heladeria artesanal'},
    {'label': 'Cervecería', 'type': 'bar', 'keyword': 'cerveceria artesanal comida'}
]

datos_lugares = []
place_ids_vistos = set()

# --- FUNCIÓN AUXILIAR PARA PROCESAR DETALLES ---
def obtener_info_lugar(p_id, etiqueta):
    try:
        detalles = gmaps.place(
            place_id=p_id, 
            fields=['name', 'geometry', 'formatted_address', 'rating', 'user_ratings_total', 'opening_hours'],
            language='es'
        ).get('result', {})

        dias_semana = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
        horarios_semanales = {dia: [] for dia in dias_semana}
        periodos = detalles.get('opening_hours', {}).get('periods', [])

        if periodos:
            for p in periodos:
                idx_google = p['open']['day']
                idx_nuestro = (idx_google - 1) % 7 
                nombre_dia = dias_semana[idx_nuestro]
                h_abre = p['open']['time']
                h_cierra = p.get('close', {}).get('time', '24h')
                horarios_semanales[nombre_dia].append({
                    'apertura': f"{h_abre[:2]}:{h_abre[2:]}",
                    'cierre': f"{h_cierra[:2]}:{h_cierra[2:]}" if h_cierra != '24h' else '24h'
                })
            for dia in dias_semana:
                if not horarios_semanales[dia]: horarios_semanales[dia] = "Cerrado"
        else:
            horarios_semanales = "No disponible"

        return {
            'nombre': detalles.get('name'),
            'tipo': etiqueta,
            'coords': detalles.get('geometry', {}).get('location'),
            'direccion': detalles.get('formatted_address'),
            'puntaje': detalles.get('rating', 0),
            'resenas': detalles.get('user_ratings_total', 0),
            'horarios_semana': horarios_semanales
        }
    except:
        return None

# --- PASO 3: PROCESAR EL ORIGEN PRIMERO ---
print("Procesando punto de origen...")
info_origen = obtener_info_lugar(origen_id, "Punto de Partida")
if info_origen:
    datos_lugares.append(info_origen)
    place_ids_vistos.add(origen_id)

# --- PASO 4: BUCLE DE BÚSQUEDA CON PAGINACIÓN ---
print(f"Buscando lugares de interés en Rosario (hasta 60 por categoría)...")
for cat in categorias_busqueda:
    print(f"Buscando {cat['label']}...")
    
    # Primera llamada (Página 1)
    busqueda = gmaps.places_nearby(
        location=location_busqueda,
        radius=3000,
        type=cat['type'],
        keyword=cat['keyword'],
        language='es'
    )

    while True:
        # Procesar resultados de la página actual
        for item in busqueda.get('results', []):
            pid = item['place_id']
            if pid in place_ids_vistos: continue
            
            # Filtro de restaurante (evitar comida al paso)
            tipos = item.get('types', [])
            if cat['label'] == 'Restaurante' and ('meal_takeaway' in tipos and 'restaurant' not in tipos):
                continue

            lugar_procesado = obtener_info_lugar(pid, cat['label'])
            if lugar_procesado:
                datos_lugares.append(lugar_procesado)
                place_ids_vistos.add(pid)

        # Verificar si hay más páginas
        token = busqueda.get('next_page_token')
        if not token:
            break
            
        # Pausa obligatoria de 2 segundos para que Google valide el token
        time.sleep(2)
        busqueda = gmaps.places_nearby(page_token=token)

# 5. Visualización
df = pd.DataFrame(datos_lugares)
print(f"\nSe encontraron {len(df) - 1} lugares + el punto de origen.")
print(df[['nombre', 'tipo', 'puntaje']].head(15))

Procesando punto de origen...
Buscando lugares de interés en Rosario (hasta 60 por categoría)...
Buscando Restaurante...
Buscando Museo...
Buscando Heladería...
Buscando Cervecería...

Se encontraron 196 lugares + el punto de origen.
                                          nombre              tipo  puntaje
0                            Terminal de omnibus  Punto de Partida      4.1
1                                   Los Jardines       Restaurante      4.3
2                             Parrilla Don Ferro       Restaurante      4.4
3                Restaurante La Farola de Maipú.       Restaurante      4.5
4   Bodegón La Atrevida - Conventillo de sabores       Restaurante      4.5
5                     El Dorado - Bodegón de Río       Restaurante      4.1
6                        Restaurante Doppio Zero       Restaurante      4.3
7                                         Riomío       Restaurante      4.2
8                            Trattoria Abruzzesa       Restaurante      4.4
9     

In [15]:
# Se guardan el dataframe completo en un archivo .txt para análisis posterior
df.to_csv('data/df_completo.txt', index=False, sep='\t')


In [1]:
import pandas as pd
import ast

# Carga del dataframe para análisis posterior
df = pd.read_csv('data/df_completo.txt', sep='\t')

# Convertimos los strings de nuevo a diccionarios
# Usamos literal_eval porque es más seguro que eval()
def safe_eval(val):
    try:
        if pd.isna(val) or val == "No disponible":
            return val
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return val

df['coords'] = df['coords'].apply(safe_eval)
df['horarios_semana'] = df['horarios_semana'].apply(safe_eval)

# Ahora ya puedes correr el código de transformación que te pasé antes
print(type(df.iloc[0]['coords'])) # Debería decir <class 'dict'>

<class 'dict'>


## Filtrado de los puntos de interés

In [23]:
# 1. Separamos el Punto de Origen (lo mantenemos siempre)
origen = df[df['tipo'] == 'Punto de Partida']

# 2. Filtramos los lugares que NO son el origen
lugares_interes = df[df['tipo'] != 'Punto de Partida'].copy()

# 3. FILTRO DE CONFIANZA: 
# Solo consideramos lugares con más de 200 reseñas (podes ajustar este número)
lugares_interes = lugares_interes[lugares_interes['resenas'] >= 200]

# 4. CREAR IDENTIFICADOR DE MARCA (Deduplicación)
# Tomamos las primeras 2 palabras del nombre para detectar sucursales (ej: "Cerveza Patagonia")
lugares_interes['marca_id'] = lugares_interes['nombre'].str.split().str[:4].str.join(' ')
# Borramos las sucursales repetidas
lugares_interes = lugares_interes.drop_duplicates(subset=['tipo', 'marca_id'], keep='first')

# --- NUEVO PASO: FILTRO DE APERTURA MIÉRCOLES ---
def esta_abierto_miercoles(horarios):
    # Si no es un diccionario o es "No disponible", lo descartamos o tratamos como cerrado
    if not isinstance(horarios, dict):
        return False
    
    status_miercoles = horarios.get('Miércoles', 'Cerrado')
    
    # Si es una lista con datos, significa que tiene turnos de apertura
    if isinstance(status_miercoles, list) and len(status_miercoles) > 0:
        return True
    
    return False

# Aplicamos el filtro
lugares_abiertos = lugares_interes[lugares_interes['horarios_semana'].apply(esta_abierto_miercoles)]
# -----------------------------------------------

# 3. Agrupamos por 'tipo' y tomamos los 5 mejores de los que SÍ abren el miércoles
top_por_categoria = lugares_abiertos.sort_values(['tipo', 'puntaje'], ascending=[True, False])
top_5_lugares = top_por_categoria.groupby('tipo').head(5)

# 4. Concatenamos el origen con la selección filtrada
df_final = pd.concat([origen, top_5_lugares]).reset_index(drop=True)

print(f"Lista final: {len(df_final)} nodos abiertos el miércoles.")
print(df_final[['tipo', 'nombre', 'horarios_semana']])

Lista final: 21 nodos abiertos el miércoles.
                tipo                                             nombre  \
0   Punto de Partida                                Terminal de omnibus   
1         Cervecería           Cerveza Patagonia - Refugio Alto Rosario   
2         Cervecería                                     Manush Rosario   
3         Cervecería                               Mosto Somos Cervezas   
4         Cervecería                             Nº40 Mercado de Birras   
5         Cervecería                              Divague Small Brewery   
6          Heladería                                         MONT BLANC   
7          Heladería                        Gelatería Italiana San Remo   
8          Heladería                                Heladería "Catania"   
9          Heladería                                         Kapricho's   
10         Heladería                                    Touche de Crème   
11             Museo          Museo de la Memoria | Ros

## Formato necesario para el optimizador de rutas

In [24]:
import numpy as np

# 1. Extraer latitud y longitud de la columna 'coords'
df_final['lat'] = df_final['coords'].apply(lambda x: x['lat'] if isinstance(x, dict) else np.nan)
df_final['lon'] = df_final['coords'].apply(lambda x: x['lng'] if isinstance(x, dict) else np.nan)

# 2. Función para extraer apertura y cierre del Miércoles
def extraer_horario_miercoles(horarios, tipo_retorno):
    # Si es "No disponible" o no es un diccionario, devolvemos NA
    if not isinstance(horarios, dict) or horarios == "No disponible":
        return "NA"
    
    # Buscamos la entrada de 'Miércoles'
    miercoles = horarios.get('Miércoles', "Cerrado")
    
    # Si el lugar está cerrado ese día
    if miercoles == "Cerrado":
        return "Cerrado"
    
    # Si tiene datos (recordá que es una lista de turnos) tomamos el primero
    if isinstance(miercoles, list) and len(miercoles) > 0:
        if tipo_retorno == 'apertura':
            return miercoles[0]['apertura']
        else:
            return miercoles[0]['cierre']
            
    return "NA"

# 3. Aplicar la función para crear las nuevas columnas
df_final['Apertura'] = df_final['horarios_semana'].apply(lambda x: extraer_horario_miercoles(x, 'apertura'))
df_final['Cierre'] = df_final['horarios_semana'].apply(lambda x: extraer_horario_miercoles(x, 'cierre'))

# 4. Crear el ID (usando el índice) y seleccionar las columnas finales
df_final['ID'] = df_final.index
columnas_interes = ['ID', 'nombre', 'tipo', 'puntaje', 'lat', 'lon', 'Apertura', 'Cierre']
df_tabla_final = df_final[columnas_interes].copy()

# 5. Visualización
print(df_tabla_final.head())

   ID                                    nombre              tipo  puntaje  \
0   0                       Terminal de omnibus  Punto de Partida      4.1   
1   1  Cerveza Patagonia - Refugio Alto Rosario        Cervecería      4.8   
2   2                            Manush Rosario        Cervecería      4.6   
3   3                      Mosto Somos Cervezas        Cervecería      4.6   
4   4                    Nº40 Mercado de Birras        Cervecería      4.6   

         lat        lon Apertura Cierre  
0 -32.939269 -60.671088    09:00  17:00  
1 -32.927238 -60.670871    12:00  00:00  
2 -32.932807 -60.652391    17:30  01:30  
3 -32.946557 -60.658378    18:00  01:00  
4 -32.948995 -60.653310    18:00  00:30  


In [25]:
# Guardamos la tabla final en un nuevo archivo .txt para análisis posterior
df_tabla_final.to_csv('data/g_nodos.txt', index=False, sep='\t')

## Cálculo matrices de distancia y tiempo

In [26]:
# Actualizamos datos lugares con los 21 lugares de interés (1 origen + 20 seleccionados)
datos_lugares = df_final.to_dict('records')

In [27]:
import pandas as pd
import time

# 1. Preparar datos de los nodos
coords_solo = [d['coords'] for d in datos_lugares]
indices_solo = [d['ID'] for d in datos_lugares]
n = len(coords_solo)

# Matrices para almacenar valores puros
matriz_segundos = []
matriz_metros = []

print(f"Calculando matriz de {n}x{n} (Valores puros: Segundos y Metros)...")

# 2. Procesamiento por filas (Batching para evitar el límite de 100 elementos)
for i in range(n):
    origen_actual = [coords_solo[i]]
    
    try:
        # Eliminamos departure_time para que NO considere tráfico
        res_fila = gmaps.distance_matrix(
            origins=origen_actual,
            destinations=coords_solo,
            mode="driving",
            language="es"
        )
        
        segundos_fila = []
        metros_fila = []
        elementos = res_fila['rows'][0]['elements']
        
        for el in elementos:
            if el['status'] == 'OK':
                # Extraemos valores puros sin procesar
                val_segundos = el['duration']['value'] # Segundos
                val_metros = el['distance']['value']   # Metros
                
                segundos_fila.append(val_segundos)
                metros_fila.append(val_metros)
            else:
                segundos_fila.append(None)
                metros_fila.append(None)
        
        matriz_segundos.append(segundos_fila)
        matriz_metros.append(metros_fila)
        
        print(f"✔ Fila {i+1}/{n} completada.")
        
    except Exception as e:
        print(f"✘ Error en fila {i}: {e}")
        matriz_segundos.append([None] * n)
        matriz_metros.append([None] * n)

# 3. Crear DataFrames con los valores crudos
df_tiempo = pd.DataFrame(matriz_segundos, index=indices_solo, columns=indices_solo)
df_distancia = pd.DataFrame(matriz_metros, index=indices_solo, columns=indices_solo)

# 4. Resultados
print("\n--- MATRIZ DE DURACIÓN (SEGUNDOS - SIN TRÁFICO) ---")
print(df_tiempo)

print("\n--- MATRIZ DE DISTANCIA (METROS) ---")
print(df_distancia)

Calculando matriz de 21x21 (Valores puros: Segundos y Metros)...
✔ Fila 1/21 completada.
✔ Fila 2/21 completada.
✔ Fila 3/21 completada.
✔ Fila 4/21 completada.
✔ Fila 5/21 completada.
✔ Fila 6/21 completada.
✔ Fila 7/21 completada.
✔ Fila 8/21 completada.
✔ Fila 9/21 completada.
✔ Fila 10/21 completada.
✔ Fila 11/21 completada.
✔ Fila 12/21 completada.
✔ Fila 13/21 completada.
✔ Fila 14/21 completada.
✔ Fila 15/21 completada.
✔ Fila 16/21 completada.
✔ Fila 17/21 completada.
✔ Fila 18/21 completada.
✔ Fila 19/21 completada.
✔ Fila 20/21 completada.
✔ Fila 21/21 completada.

--- MATRIZ DE DURACIÓN (SEGUNDOS - SIN TRÁFICO) ---
     0     1    2    3    4    5    6     7     8     9   ...   11   12   13  \
0     0   382  501  532  592  382  782   531   387   610  ...  605  612  770   
1   588     0  421  688  749  426  966   907   773  1002  ...  773  796  955   
2   452   479    0  396  464  117  763   757   764  1035  ...  456  593  751   
3   370   724  393    0  206  294  491   550  

In [28]:
df_tiempo

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0,382,501,532,592,382,782,531,387,610,...,605,612,770,798,632,796,380,324,323,775
1,588,0,421,688,749,426,966,907,773,1002,...,773,796,955,807,448,980,332,437,369,783
2,452,479,0,396,464,117,763,757,764,1035,...,456,593,751,572,233,765,478,193,219,548
3,370,724,393,0,206,294,491,550,557,834,...,224,321,479,686,516,517,723,253,267,646
4,502,879,441,220,0,438,301,578,564,846,...,133,187,351,588,526,316,878,429,459,570
5,455,586,119,377,332,0,624,762,769,1059,...,339,521,677,491,250,637,584,190,217,467
6,846,1199,802,559,491,799,0,623,749,1000,...,494,341,448,597,887,227,1198,790,785,561
7,454,835,707,405,572,602,451,0,447,680,...,575,282,287,1010,825,466,834,527,540,987
8,415,479,704,482,648,585,653,355,0,382,...,652,483,642,1059,835,668,478,583,525,1036
9,596,660,867,652,818,748,851,598,356,0,...,822,681,840,1240,998,866,659,743,688,1216


In [29]:
df_distancia

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0,2020,2851,2636,2968,2309,4533,2615,1848,3484,...,3035,3447,4487,4181,3346,4625,2015,1797,1883,4150
1,2862,0,2478,3827,4221,2485,5727,5056,4401,5670,...,4355,4641,5681,4671,2472,5818,1730,2478,2010,4640
2,2596,2687,0,1857,2245,511,3806,4176,4446,5903,...,2308,2719,3759,2908,1022,3889,2682,1029,1130,2876
3,1898,4061,1900,0,909,1356,2469,2907,3177,4647,...,1042,1383,2422,3225,2392,2552,4056,1352,1467,2869
4,2867,5192,2344,991,0,2325,1554,2756,2975,5315,...,716,855,1839,2901,2571,1645,5187,2318,2457,2531
5,2579,3228,542,1838,1728,0,3284,4154,4424,5899,...,1797,2283,3266,2400,1037,3375,3223,1012,1126,2369
6,4709,7198,4176,2812,2410,4158,0,3173,4298,6120,...,2548,1660,2366,2622,4404,1145,7193,4151,4283,2504
7,2603,4623,4148,2244,3155,3605,2830,0,2585,4260,...,3293,1744,1933,5477,4640,2921,4618,3093,3208,5446
8,2178,2900,4407,2652,3563,3865,4036,1865,0,2328,...,3701,2950,3990,5877,4902,4127,2895,3498,3439,5846
9,3247,3970,5575,3722,4633,5033,5516,3927,1971,0,...,4771,4430,5469,6947,6070,5607,3964,4696,4607,6916


In [30]:
# Guardar df_distancias en un archivo CSV
df_distancia.to_csv('data/g_distancias.csv')

# Guardar df_tiempos en un archivo CSV
df_tiempo.to_csv('data/g_tiempos.csv')
